# Multi-label Topic Classification for HEK App Reviews

This notebook fine-tunes pretrained Transformer models for multi-label topic classification on manually labeled HEK Service-App reviews. The labeled subset is used to train and evaluate the models.

The notebook performs the following steps:

1. Environment and library initialization.
2. Loading of labeled data and construction of a multi-label target space.
3. Train–validation split of the labeled reviews.
4. Conversion to Hugging Face `Dataset` format and tokenization.
5. Configuration and training of two German-capable Transformer models.
6. Quantitative evaluation on the validation split (micro/macro F1 and per-label metrics).


## Environment setup and imports

This section initializes core libraries for data handling, model training, and evaluation.


In [ ]:
# Core libraries for data handling and evaluation
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, f1_score, classification_report

# PyTorch backend for the Transformer models
import torch
from torch import nn

# Hugging Face Transformers and Datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

## Load labeled data and construct label space

The labeled Excel file contains manually annotated topics per review. This section loads the labeled dataset, derives the set of unique topic labels, and encodes each review’s topics as a multi-hot label vector that can be used for multi-label classification.


In [ ]:
# Paths to labeled (manual annotations) and unlabeled data
LABELED_PATH = '../data/labeling/manual_labeling_pool_hek_viactiv.xlsx'
UNLABELED_PATH = '../data/labeling/training_pool_hek_viactiv.xlsx'

labeled_df = pd.read_excel(LABELED_PATH)
unlabeled_df = pd.read_excel(UNLABELED_PATH)

print('Labeled rows:', len(labeled_df))
print('Unlabeled rows:', len(unlabeled_df))

# Split semicolon-separated topic string into list of topic names
labeled_df['topics'] = labeled_df['label_topics_raw'].astype(str).str.split(';')

# Derive global label set and mappings
all_labels = sorted({t.strip() for ts in labeled_df['topics'] for t in ts if t.strip()})
label2id = {lbl: i for i, lbl in enumerate(all_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}

num_labels = len(all_labels)
print(f'Labels ({num_labels}):', all_labels)

# Multi-hot encode labels for each review
def encode_labels(topic_list):
    vec = np.zeros(num_labels, dtype=np.float32)
    for t in topic_list:
        t = t.strip()
        if t in label2id:
            vec[label2id[t]] = 1.0
    return vec

labeled_df['label_vec'] = labeled_df['topics'].apply(encode_labels)

## Train–validation split

The labeled dataset is split into training and validation sets. The split is random but reproducible due to a fixed random seed.


In [ ]:
train_df, val_df = train_test_split(
    labeled_df,
    test_size=0.2,
    random_state=42,
)

print('Train rows:', len(train_df), 'Val rows:', len(val_df))

## Conversion to Hugging Face Datasets and tokenization

The pandas DataFrames are converted to Hugging Face `Dataset` objects. Tokenization prepares model inputs (`input_ids`, `attention_mask`) from the review texts with a fixed maximum sequence length.


In [ ]:
TEXT_COL = 'review_text'
ID_COL = 'review_id' if 'review_id' in labeled_df.columns else None

train_keep_cols = [TEXT_COL, 'label_vec']
val_keep_cols = [TEXT_COL, 'label_vec']
unlabeled_keep_cols = [TEXT_COL]

if ID_COL is not None:
    train_keep_cols.append(ID_COL)
    val_keep_cols.append(ID_COL)
    if ID_COL in unlabeled_df.columns:
        unlabeled_keep_cols.append(ID_COL)

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
unlabeled_df = unlabeled_df.reset_index(drop=True).copy()

if ID_COL is None:
    train_df['row_id'] = [f"train_{i}" for i in range(len(train_df))]
    val_df['row_id'] = [f"val_{i}" for i in range(len(val_df))]
    unlabeled_df['row_id'] = [f"unlabeled_{i}" for i in range(len(unlabeled_df))]
    ID_COL = 'row_id'
    train_keep_cols.append(ID_COL)
    val_keep_cols.append(ID_COL)
    unlabeled_keep_cols.append(ID_COL)

train_dataset = Dataset.from_pandas(train_df[train_keep_cols])
val_dataset = Dataset.from_pandas(val_df[val_keep_cols])
unlabeled_dataset = Dataset.from_pandas(unlabeled_df[unlabeled_keep_cols])

# Model identifiers
roberta_model_name = 'xlm-roberta-base'  # Multilingual RoBERTa-family model with German support
german_app_model_name = 'oliverguhr/german-sentiment-bert'  # German BERT, trained on multiple domains including app reviews

max_length = 256


def tokenize_function(examples, tokenizer):
    return tokenizer(
        examples[TEXT_COL],
        padding='max_length',
        truncation=True,
        max_length=max_length,
    )

## Model configuration and training setup

This section defines a helper function that constructs a `Trainer` instance for a given pretrained model. The configuration uses a multi-label classification head and binary cross-entropy loss (`BCEWithLogitsLoss`). Evaluation metrics are micro- and macro-averaged F1 scores on the validation set.


In [ ]:
def make_trainer(model_name, output_dir_suffix):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenize datasets
    tokenized_train = train_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
    )
    tokenized_val = val_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
    )

    # Add labels as separate tensor field for Trainer
    tokenized_train = tokenized_train.add_column('labels', list(train_df['label_vec'].values))
    tokenized_val = tokenized_val.add_column('labels', list(val_df['label_vec'].values))

    torch_cols = ['input_ids', 'attention_mask', 'labels']
    if 'token_type_ids' in tokenized_train.column_names:
        torch_cols.append('token_type_ids')

    tokenized_train.set_format(type='torch', columns=torch_cols)
    tokenized_val.set_format(type='torch', columns=torch_cols)

    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type='multi_label_classification',
        id2label=id2label,
        label2id=label2id,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,  # allow new classifier head with different shape
    )

    # Use BCEWithLogitsLoss for multi-label classification
    class MultiLabelTrainer(Trainer):
        def compute_loss(
            self,
            model,
            inputs,
            return_outputs=False,
            num_items_in_batch=None,  # accept extra arg used by your Trainer
        ):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.float())
            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs > 0.5).astype(int)
        micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        return {'micro_f1': micro_f1, 'macro_f1': macro_f1}

    output_dir = f'./model_{output_dir_suffix}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',      # <-- changed name
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='micro_f1',
        greater_is_better=True,
        logging_steps=50,
    )

    trainer = MultiLabelTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    return trainer, tokenizer

def find_optimal_thresholds(probs, labels_true, label_names, min_positives=3):
    """
    probs: np.ndarray of shape (n_samples, n_labels) with sigmoid probabilities
    labels_true: np.ndarray of shape (n_samples, n_labels) with 0/1 ground truth
    label_names: list of label names in the same order as columns
    min_positives: minimum number of positive examples required to tune a threshold
    """
    thresholds = {}
    
    # 1. Define the labels that MUST be visible on the dashboard for product management
    business_critical_labels = ['document_management', 'smarthealth_epa_features', 'customer_service', 'updates_versions', 'usability_ui']
    
    for i, name in enumerate(label_names):
        y_true = labels_true[:, i]
        y_scores = probs[:, i]

        # If too few positives, fall back to a default threshold
        if y_true.sum() < min_positives:
            thresholds[name] = 0.5
            continue

        best_thr = 0.5
        best_f1 = 0.0

        # Simple grid search over thresholds starting at 0.25
        for thr in np.linspace(0.25, 0.9, 17):
            y_pred = (y_scores >= thr).astype(int)
            
            f1 = f1_score(y_true, y_pred, zero_division=0)
            precision = precision_score(y_true, y_pred, zero_division=0)
            
            # --- THE HYBRID LOGIC ---
            if name in business_critical_labels:
                # For critical labels: maximize F1 without the strict 50% precision block
                if f1 > best_f1:
                    best_f1 = f1
                    best_thr = thr
            else:
                # For all other labels: Precision MUST be >= 0.5 (50%)
                if precision >= 0.5 and f1 > best_f1:
                    best_f1 = f1
                    best_thr = thr

        # --- SAFETY NET ---
        # Prevent business critical labels from dropping below 0.25 (which causes too much noise)
        if name in business_critical_labels and best_thr < 0.25:
            thresholds[name] = 0.25
        else:
            thresholds[name] = best_thr

    return thresholds

def predict_topics(texts, model, tokenizer, thresholds, label_names, max_length=256):
    model.eval()
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.sigmoid(logits).cpu().numpy()

    all_preds = []
    for row in probs:
        labels_for_row = []
        for i, name in enumerate(label_names):
            thr = thresholds.get(name, 0.5)
            if row[i] >= thr:
                labels_for_row.append(name)
        all_preds.append(labels_for_row)
    return all_preds

In [ ]:
def get_val_predictions(trainer, val_df, label_names, text_col=TEXT_COL, id_col=ID_COL):
    pred_output = trainer.predict(trainer.eval_dataset)

    logits = pred_output.predictions
    y_true = pred_output.label_ids
    probs = 1 / (1 + np.exp(-logits))

    out_df = val_df[[text_col, id_col]].copy().reset_index(drop=True)

    for i, label in enumerate(label_names):
        out_df[f'true_{label}'] = y_true[:, i].astype(int)
        out_df[f'prob_{label}'] = probs[:, i]

    return out_df, y_true, probs


def calibrate_thresholds(probs, y_true, label_names, grid=None, optimize='f1'):
    if grid is None:
        grid = np.arange(0.05, 0.95, 0.05)

    best_thresholds = {}
    best_label_scores = {}

    for i, label in enumerate(label_names):
        y_label = y_true[:, i].astype(int)
        p_label = probs[:, i]

        best_thr = 0.5
        best_score = -1.0

        for thr in grid:
            pred_label = (p_label >= thr).astype(int)

            if optimize == 'f1':
                score = f1_score(y_label, pred_label, zero_division=0)
            else:
                raise ValueError("optimize must be 'f1'")

            if score > best_score:
                best_score = score
                best_thr = float(thr)

        best_thresholds[label] = best_thr
        best_label_scores[label] = best_score

    return best_thresholds, best_label_scores


def apply_thresholds(probs, label_names, thresholds):
    preds = np.zeros_like(probs, dtype=int)
    for i, label in enumerate(label_names):
        thr = thresholds.get(label, 0.5)
        preds[:, i] = (probs[:, i] >= thr).astype(int)
    return preds


def labels_from_binary_matrix(matrix, label_names):
    return [
        [label_names[j] for j in range(len(label_names)) if row[j] == 1]
        for row in matrix
    ]


def build_prediction_export_df(base_df, y_true, probs, preds, label_names, text_col=TEXT_COL, id_col=ID_COL):
    export_df = base_df[[text_col, id_col]].reset_index(drop=True).copy()

    y_true = np.asarray(y_true, dtype=np.int8)
    preds = np.asarray(preds, dtype=np.int8)

    label_arr = np.asarray(label_names)

    true_label_lists = [label_arr[row.astype(bool)] for row in y_true]
    pred_label_lists = [label_arr[row.astype(bool)] for row in preds]

    fp_matrix = (preds == 1) & (y_true == 0)
    fn_matrix = (y_true == 1) & (preds == 0)

    export_df["true_labels"] = [";".join(x) for x in true_label_lists]
    export_df["predicted_labels"] = [";".join(x) for x in pred_label_lists]
    export_df["false_positives"] = [";".join(label_arr[row]) for row in fp_matrix]
    export_df["false_negatives"] = [";".join(label_arr[row]) for row in fn_matrix]
    export_df["n_false_positives"] = fp_matrix.sum(axis=1).astype(int)
    export_df["n_false_negatives"] = fn_matrix.sum(axis=1).astype(int)

    return export_df


def export_error_files(base_df, y_true, probs, preds, label_names, model_tag, text_col=TEXT_COL, id_col=ID_COL):
    export_df = build_prediction_export_df(
        base_df=base_df,
        y_true=y_true,
        probs=probs,
        preds=preds,
        label_names=label_names,
        text_col=text_col,
        id_col=id_col,
    )

    false_positives_df = export_df[export_df["n_false_positives"] > 0].copy()
    false_negatives_df = export_df[export_df["n_false_negatives"] > 0].copy()

    output_dir = "./output/02_model_selection"
    os.makedirs(output_dir, exist_ok=True)

    false_positives_path = f"{output_dir}/{model_tag}_false_positives.xlsx"
    false_negatives_path = f"{output_dir}/{model_tag}_false_negatives.xlsx"
    false_positives_path_csv = f"{output_dir}/{model_tag}_false_positives.csv"
    false_negatives_path_csv = f"{output_dir}/{model_tag}_false_negatives.csv"

    false_positives_df.to_excel(false_positives_path, index=False)
    false_negatives_df.to_excel(false_negatives_path, index=False)
    false_positives_df.to_csv(false_positives_path_csv, index=False)
    false_negatives_df.to_csv(false_negatives_path_csv, index=False)

    print(f"Saved: {false_positives_path}")
    print(f"Saved: {false_negatives_path}")

    return export_df, false_positives_df, false_negatives_df


def print_thresholded_metrics(model_name, y_true, preds, label_names):
    micro_f1 = f1_score(y_true, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, preds, average='macro', zero_division=0)

    print(f"{model_name} calibrated micro F1: {micro_f1:.3f}")
    print(f"{model_name} calibrated macro F1: {macro_f1:.3f}")
    print(f"Per-label classification report ({model_name}, calibrated):")
    print(classification_report(y_true, preds, target_names=label_names, zero_division=0))

    return micro_f1, macro_f1

## Training and validation evaluation

Both models are trained for several epochs on the training split. After training, each model is evaluated on the held-out validation set. The Hugging Face `Trainer` computes micro and macro F1 scores based on a 0.5 probability threshold per label.


In [ ]:
# Train XLM-RoBERTa model
roberta_trainer, roberta_tokenizer = make_trainer(roberta_model_name, 'xlm-roberta-base')
roberta_train_result = roberta_trainer.train()
print(roberta_train_result)
roberta_metrics = roberta_trainer.evaluate()
print('XLM-RoBERTa metrics:', roberta_metrics)

# Train German BERT model (pretrained on app and other reviews)
german_trainer, german_tokenizer = make_trainer(german_app_model_name, 'german-sentiment-bert')
german_train_result = german_trainer.train()
print(german_train_result)
german_metrics = german_trainer.evaluate()
print('German BERT metrics:', german_metrics)

## Detailed evaluation: per-label metrics

To better understand model behavior, additional evaluation computes micro and macro F1 scores and a `classification_report` with per-label precision, recall, and F1 on the validation set for both models.


In [ ]:
# --- XLM-RoBERTa with per-label thresholds ---
roberta_eval_outputs = roberta_trainer.predict(roberta_trainer.eval_dataset)
roberta_logits = roberta_eval_outputs.predictions
roberta_labels_true = roberta_eval_outputs.label_ids

roberta_probs = 1 / (1 + np.exp(-roberta_logits))

# learn thresholds per label
roberta_thresholds = find_optimal_thresholds(
    probs=roberta_probs,
    labels_true=roberta_labels_true,
    label_names=all_labels,
    min_positives=3,
)

# apply thresholds
roberta_preds = np.zeros_like(roberta_probs, dtype=int)
for i, name in enumerate(all_labels):
    thr = roberta_thresholds[name]
    roberta_preds[:, i] = (roberta_probs[:, i] >= thr).astype(int)

roberta_micro_f1 = f1_score(roberta_labels_true, roberta_preds, average='micro', zero_division=0)
roberta_macro_f1 = f1_score(roberta_labels_true, roberta_preds, average='macro', zero_division=0)
print('XLM-RoBERTa thresholds:', roberta_thresholds)
print(f'XLM-RoBERTa micro F1: {roberta_micro_f1:.3f}')
print(f'XLM-RoBERTa macro F1: {roberta_macro_f1:.3f}')

print('Per-label classification report (XLM-RoBERTa):')
print(classification_report(
    roberta_labels_true,
    roberta_preds,
    target_names=all_labels,
    zero_division=0,
))

# --- German BERT with per-label thresholds ---
german_eval_outputs = german_trainer.predict(german_trainer.eval_dataset)
german_logits = german_eval_outputs.predictions
german_labels_true = german_eval_outputs.label_ids

german_probs = 1 / (1 + np.exp(-german_logits))

german_thresholds = find_optimal_thresholds(
    probs=german_probs,
    labels_true=german_labels_true,
    label_names=all_labels,
    min_positives=3,
)

german_preds = np.zeros_like(german_probs, dtype=int)
for i, name in enumerate(all_labels):
    thr = german_thresholds[name]
    german_preds[:, i] = (german_probs[:, i] >= thr).astype(int)

german_micro_f1 = f1_score(german_labels_true, german_preds, average='micro', zero_division=0)
german_macro_f1 = f1_score(german_labels_true, german_preds, average='macro', zero_division=0)
print('German BERT thresholds:', german_thresholds)
print(f'German BERT micro F1: {german_micro_f1:.3f}')
print(f'German BERT macro F1: {german_macro_f1:.3f}')

print('Per-label classification report (German BERT):')
print(classification_report(
    german_labels_true,
    german_preds,
    target_names=all_labels,
    zero_division=0,
))

thresholds = german_thresholds  # or roberta_thresholds
label_names = all_labels

In [ ]:
roberta_export_df, roberta_fp_df, roberta_fn_df = export_error_files(
    base_df=val_df,
    y_true=roberta_labels_true,
    probs=roberta_probs,
    preds=roberta_preds,
    label_names=all_labels,
    model_tag="xlm_roberta",
    text_col=TEXT_COL,
    id_col=ID_COL,
)

german_export_df, german_fp_df, german_fn_df = export_error_files(
    base_df=val_df,
    y_true=german_labels_true,
    probs=german_probs,
    preds=german_preds,
    label_names=all_labels,
    model_tag="german_sentiment_bert",
    text_col=TEXT_COL,
    id_col=ID_COL,
)